# LMIP Ingestion Layer - Complete Documentation

The Ingestion layer is the **data acquisition gateway** of the Labour Market Intelligence Platform. It handles API authentication, rate limiting, pagination, and reliable data extraction from multiple job posting sources.

---

## Architecture Principles

* **Source Abstraction**: Unified interface for heterogeneous APIs
* **Resilience**: Automatic retry with exponential backoff
* **Rate Limiting**: Respect API quotas and avoid bans
* **Incremental Loading**: Fetch only new/updated records when possible
* **Payload Preservation**: Store complete raw API responses for reprocessing
* **Audit Trail**: Track all ingestion runs with success/failure metrics

---

## Pipeline Flow

```
┌─────────────────────────────────────────────────────────────────┐
│                    EXTERNAL JOB BOARDS                          │
│                   Remotive, Arbeitnow, etc.                     │
└────────────────────┬────────────────────────────────────────────┘
                     │
        ┌────────────┴────────────┐
        ▼                         ▼
┌─────────────────────┐  ┌─────────────────────┐
│  1a. SOURCE INGESTOR│  │  1b. SOURCE INGESTOR│
│  ingest_remotive    │  │  ingest_arbeitnow   │
│  • Fetch API data   │  │  • Fetch API data   │
│  • Validate records │  │  • Validate records │
│  • Extract job IDs  │  │  • Extract job IDs  │
│  • Use helpers      │  │  • Use helpers      │
└──────────┬──────────┘  └──────────┬──────────┘
           │                        │
           └────────┬───────────────┘
                    ▼
┌─────────────────────────────────────────────────────────────────┐
│  2. COMMON HELPERS (ingest_common_helpers)                      │
│     • Generate IDs and hashes                                   │
│     • Prepare bronze snapshots                                  │
│     • Main ingestion orchestration                              │
│     ➜ Shared by all source ingestors                            │
└────────────────────┬────────────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────────────┐
│  3. MANIFEST WRITER (ingest_manifest_writer)                    │
│     • Log API responses                                         │
│     • Track pipeline runs                                       │
│     • Audit logging                                             │
│     ➜ bronze.bronze_api_response_log                            │
│     ➜ metadata.pipeline_run_control                             │
│     ➜ audit.audit_pipeline_runs                                 │
└────────────────────┬────────────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────────────┐
│  4. BRONZE LAYER OUTPUT                                         │
│     ➜ bronze.bronze_job_snapshot (raw job data)                 │
└─────────────────────────────────────────────────────────────────┘
```

---

## Ingestion Components

### 1. ingest_common_helpers

**Purpose**: Shared utilities used by all source ingestors

**Key Functions**:

#### ID Generation & Hashing
- `generate_batch_id()`: Create unique UUID for each ingestion run
- `generate_snapshot_id(source, job_id, batch)`: Create unique snapshot identifier
- `generate_payload_hash(json)`: SHA-256 hash for deduplication
- `extract_arbeitnow_job_id(job)`: Extract slug from Arbeitnow records
- `extract_remotive_job_id(job)`: Extract ID from Remotive records

#### Bronze Preparation
- `prepare_bronze_snapshots(jobs, source, batch, extract_func)`: Transform API responses into bronze schema
  - Adds snapshot metadata (timestamp, hash, IDs)
  - Partitions by source and date
  - Standardizes structure across sources

#### Main Orchestration
- `ingest_to_bronze(source, fetch_func, url, extract_func, ...)`: Core ingestion flow
  1. Start pipeline run in metadata
  2. Fetch data from API
  3. Log API response
  4. Prepare bronze snapshots
  5. Write to bronze.bronze_job_snapshot
  6. Complete pipeline run with audit log
  - Returns: (success, batch_id, record_count)

**Configuration**:
```python
CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
METADATA_SCHEMA = "metadata"
AUDIT_SCHEMA = "audit"

BRONZE_JOB_SNAPSHOT = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_job_snapshot"
BRONZE_API_LOG = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_api_response_log"
PIPELINE_RUN_CONTROL = f"{CATALOG}.{METADATA_SCHEMA}.pipeline_run_control"
AUDIT_PIPELINE_RUNS = f"{CATALOG}.{AUDIT_SCHEMA}.audit_pipeline_runs"
```

---

### 2. ingest_manifest_writer

**Purpose**: Metadata and audit logging for pipeline runs

**Key Functions**:

#### API Response Logging
- `log_api_response(source, batch, url, status, retry, rate_limit, time_ms)`
  - Writes to `bronze.bronze_api_response_log`
  - Tracks HTTP status, response time, retries
  - Flags rate limit hits

#### Pipeline Run Control
- `start_pipeline_run(name, source, batch)`: Register run start
  - Writes to `metadata.pipeline_run_control`
  - Sets status to RUNNING
  - Returns run_control_sk

- `complete_pipeline_run(batch, status)`: Update run completion
  - Sets ended_at timestamp
  - Updates status (SUCCESS/FAILED)

#### Audit Logging
- `log_audit_pipeline_run(run_id, name, status, rows_read, rows_written, runtime, error)`
  - Writes to `audit.audit_pipeline_runs`
  - Comprehensive run metrics
  - Error details if failed

**Schemas Defined**:
- `API_LOG_SCHEMA`: API response log structure
- `PIPELINE_RUN_SCHEMA`: Pipeline run control structure
- `AUDIT_RUN_SCHEMA`: Audit log structure

---

### 3. Source-Specific Ingestors

Each source has a dedicated notebook that:
1. Imports common helpers and manifest writer
2. Defines source-specific configuration
3. Implements API fetch function
4. Validates record structure
5. Executes ingestion via shared helpers

#### ingest_remotive

**API Details**:
- **Endpoint**: https://remotive.com/api/remote-jobs
- **Auth**: None (public API)
- **Rate Limit**: No official limit
- **Pagination**: None (returns all jobs)

**Key Features**:
- Remote-first job board
- Tech and non-tech positions
- Company logos included
- Tags and categories

**Field Mapping**:
```python
REMOTIVE_FIELDS = {
    'job_id': 'id',                    # int64
    'title': 'title',
    'company': 'company_name',
    'location': 'candidate_required_location',
    'publication_date': 'publication_date',  # ISO date string
    'description': 'description',
    'url': 'url',
    'tags': 'tags',                    # array/object
    'job_type': 'job_type',
    'salary': 'salary'
}
```

**Validation**:
- Required fields: `id`, `title`, `company_name`, `url`
- Type checks: `id` must be int64
- Date format validation for `publication_date`

**Recent Results**: 23 jobs ingested in 6.74s (2026-06-03)

---

#### ingest_arbeitnow

**API Details**:
- **Endpoint**: https://www.arbeitnow.com/api/job-board-api
- **Auth**: None (public API)
- **Rate Limit**: No official limit
- **Pagination**: None (returns all jobs)

**Key Features**:
- European job board focus
- Remote and hybrid positions
- Multiple job types per posting
- Unix timestamp for posting dates

**Field Mapping**:
```python
ARBEITNOW_FIELDS = {
    'job_id': 'slug',              # string - unique slug
    'title': 'title',
    'company': 'company_name',
    'location': 'location',
    'remote': 'remote',            # boolean
    'url': 'url',
    'tags': 'tags',                # array/object
    'job_types': 'job_types',      # array/object
    'description': 'description',
    'created_at': 'created_at'     # int64 - Unix timestamp
}
```

**Validation**:
- Required fields: `slug`, `title`, `company_name`
- Type checks: `created_at` must be int64, `remote` must be boolean
- Skips invalid records with warnings

**Recent Results**: 100 jobs ingested in 6.03s (2026-06-03)

---

## Configuration

### Current Data Sources

**Active Sources**:
1. **Remotive** (remotive.com) - Remote job board, 23 jobs per run
2. **Arbeitnow** (arbeitnow.com) - European job board, ~100 jobs per run

**Future Sources** (add following same pattern):
- LinkedIn, Indeed, Glassdoor, GitHub Jobs, etc.
- Any source with public API or data feed

### Bronze Layer Output Schema

All sources write to `bronze.bronze_job_snapshot` with this structure:

```python
bronze_schema = StructType([
    StructField("snapshot_id", StringType(), False),      # PK: source_jobid_batch
    StructField("source_name", StringType(), False),     # remotive, arbeitnow, etc.
    StructField("source_job_id", StringType(), True),    # Original API job ID
    StructField("batch_id", StringType(), False),        # UUID batch identifier
    StructField("page_number", IntegerType(), True),     # For paginated APIs
    StructField("page_size", IntegerType(), True),       # Records per page
    StructField("payload_json", StringType(), False),    # Complete API response
    StructField("payload_hash", StringType(), False),    # SHA-256 for dedup
    StructField("ingestion_timestamp", TimestampType(), False),
    StructField("ingestion_date", DateType(), False),    # Partition key
    StructField("source_status_code", IntegerType(), True),  # HTTP status
    StructField("source_etag", StringType(), True)       # ETag if available
])
```

**Partitioning**: By `source_name` and `ingestion_date`

### Metadata & Audit Tables

#### bronze.bronze_api_response_log
Tracks every API call:
- Response time, HTTP status, retry count
- Rate limit flags
- Timestamp per request

#### metadata.pipeline_run_control
Pipeline execution control:
- Run status (RUNNING, SUCCESS, FAILED)
- Start/end timestamps
- Trigger type (MANUAL, SCHEDULED)

#### audit.audit_pipeline_runs
Comprehensive audit trail:
- Rows read/written
- Runtime seconds
- Error messages
- Environment (PROD, DEV)

---

## Execution Guide

### Running Individual Sources

Each source notebook can be run directly or via `dbutils.notebook.run`:

#### Remotive Ingestion

```python
# Run directly in notebook
dbutils.notebook.run(
    "/LMIP/notebooks/ingestion/ingest_remotive",
    timeout_seconds=300
)
```

**Expected Output**:
- ~20-30 remote jobs per run
- Duration: 5-10 seconds
- No parameters needed (full refresh each time)

---

#### Arbeitnow Ingestion

```python
# Run directly in notebook
dbutils.notebook.run(
    "/LMIP/notebooks/ingestion/ingest_arbeitnow",
    timeout_seconds=300
)
```

**Expected Output**:
- ~100 jobs per run
- Duration: 5-10 seconds
- No parameters needed (full refresh each time)

---

### Monitoring Ingestion

#### Check Recent Snapshots

```sql
-- View latest bronze snapshots by source
SELECT 
  source_name,
  batch_id,
  COUNT(*) as record_count,
  MIN(ingestion_timestamp) as first_ingested,
  MAX(ingestion_timestamp) as last_ingested
FROM bronze.bronze_job_snapshot
GROUP BY source_name, batch_id
ORDER BY last_ingested DESC
LIMIT 20;
```

#### Check Pipeline Audit History

```sql
-- View all pipeline runs with metrics
SELECT 
  run_id as batch_id,
  pipeline_name,
  status,
  start_time,
  end_time,
  rows_read,
  rows_written,
  ROUND(runtime_seconds, 2) as runtime_sec,
  error_message
FROM audit.audit_pipeline_runs
WHERE pipeline_name LIKE 'bronze_ingestion_%'
ORDER BY start_time DESC
LIMIT 20;
```

#### Check API Response Telemetry

```sql
-- View API performance metrics
SELECT 
  source_name,
  batch_id,
  http_status_code,
  response_time_ms,
  retry_count,
  rate_limit_hit,
  logged_at
FROM bronze.bronze_api_response_log
ORDER BY logged_at DESC
LIMIT 20;
```

---

### Scheduled Execution

Create a Databricks Job to run both ingestors daily:

```python
# Job configuration (conceptual)
tasks = [
    {
        "task_key": "ingest_remotive",
        "notebook_path": "/LMIP/notebooks/ingestion/ingest_remotive",
        "timeout_seconds": 300
    },
    {
        "task_key": "ingest_arbeitnow",
        "notebook_path": "/LMIP/notebooks/ingestion/ingest_arbeitnow",
        "timeout_seconds": 300
    }
]

# Schedule: Daily at 6 AM UTC
schedule = {"quartz_cron_expression": "0 0 6 * * ?", "timezone_id": "UTC"}
```

**Recommendation**: Run daily to catch new postings

---

## Adding a New Source

Follow this pattern to add any new job board:

### Step 1: Create Source Ingestor Notebook

Create `ingest_<source_name>.ipynb`:

```python
# Cell 1: Import helpers
%run ./ingest_common_helpers
%run ./ingest_manifest_writer

# Cell 2: Configuration
SOURCE_NAME = "<source_name>"
API_URL = "<api_endpoint>"

# Define field mapping
SOURCE_FIELDS = {
    'job_id': '<api_id_field>',
    'title': '<api_title_field>',
    'company': '<api_company_field>',
    # ... other fields
}

# Cell 3: Extract function
def extract_<source>_job_id(job):
    """Extract unique job ID from API response"""
    return job.get('<id_field>', '')

# Cell 4: Validation function
def validate_<source>_record(job):
    """Validate required fields and types"""
    required_fields = ['<field1>', '<field2>']
    for field in required_fields:
        if not job.get(field):
            return False, f"Missing: {field}"
    return True, None

# Cell 5: Fetch function
def fetch_<source>_jobs():
    """Fetch jobs from API"""
    try:
        response = requests.get(API_URL, timeout=30)
        response.raise_for_status()
        data = response.json()
        jobs = data.get('<jobs_key>', [])
        
        # Validate
        validated = []
        for job in jobs:
            is_valid, error = validate_<source>_record(job)
            if is_valid:
                validated.append(job)
            else:
                print(f"WARNING: Skipping - {error}")
        
        return validated, None
    except Exception as e:
        return None, str(e)

# Cell 6: Execute ingestion
success, batch_id, count = ingest_to_bronze(
    source_name=SOURCE_NAME,
    fetch_function=fetch_<source>_jobs,
    api_url=API_URL,
    extract_job_id_func=extract_<source>_job_id,
    log_api_response_func=log_api_response,
    start_pipeline_run_func=start_pipeline_run,
    complete_pipeline_run_func=complete_pipeline_run,
    log_audit_func=log_audit_pipeline_run
)

print(f"Status: {'SUCCESS' if success else 'FAILED'}")
if success:
    print(f"Batch: {batch_id}, Records: {count}")
```

### Step 2: Test the Ingestor

1. Run the notebook manually
2. Check `bronze.bronze_job_snapshot` for records
3. Verify `audit.audit_pipeline_runs` shows SUCCESS
4. Inspect API logs in `bronze.bronze_api_response_log`

### Step 3: Add to Schedule

Add the new source to your daily job:

```python
tasks.append({
    "task_key": "ingest_<source>",
    "notebook_path": "/LMIP/notebooks/ingestion/ingest_<source>",
    "timeout_seconds": 300
})
```

---

## Best Practices

### Error Handling

1. **Validation First**: Check record structure before writing
2. **Graceful Degradation**: Skip invalid records, log warnings
3. **Audit Everything**: Use manifest writer for all operations
4. **Timeout Safety**: Set reasonable timeouts (30s for API calls)

### Performance

1. **Small Batch Size**: Current sources return 20-100 jobs (fast)
2. **No Pagination Needed**: Most APIs return full dataset
3. **Parallel Execution**: Run sources in parallel via Jobs
4. **UTC Timestamps**: Standardize all dates to UTC

### Data Quality

1. **Payload Hash**: SHA-256 for deduplication in bronze
2. **Required Field Validation**: Enforce minimums per source
3. **Type Checking**: Validate int64, bool, string types
4. **Full Payload Storage**: Store complete API response

---

## Troubleshooting

### Issue: Zero Records Ingested

**Check**:
1. API endpoint responding? (test with curl)
2. Response structure changed?
3. Validation too strict?

**Debug**:
```python
# Test API manually
import requests
resp = requests.get(API_URL)
print(resp.status_code)
print(resp.json())
```

---

### Issue: Invalid Records Skipped

**Symptoms**: Warnings during ingestion

**Fix**: Update validation function to match actual API response:
```python
# Check what fields API actually returns
for job in jobs[:1]:  # First record
    print(job.keys())
```

---

### Issue: Duplicate Snapshots

**Root Cause**: Same job ingested multiple times

**Prevention**:
- `payload_hash` in bronze enables deduplication
- Silver layer handles cross-batch dedup
- Use `source_job_id` + `source_name` as natural key

---

## Next Steps: Bronze Layer

Ingested data flows to the **Bronze layer** for:

1. **Deduplication**: Remove duplicate snapshots
2. **Versioning**: Track job posting updates over time
3. **Payload Indexing**: Extract searchable fields from JSON
4. **Retention Management**: Archive old snapshots

See [bronze/README_BRONZE](../bronze/README_BRONZE) for details.
